In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

# **configuration**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from typing import List
from pyspark.sql import DataFrame
from pyspark.sql.window import Window

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [0]:
import os
import sys

In [0]:
current_dir = os.getcwd()

sys.path.append(current_dir)

In [0]:
current_dir

'/Workspace/Users/ghayda8tebessi@gmail.com/pysparkdbt_project'

In [0]:
import utils.util_tran as cu
dir(cu)

['DataFrame',
 'DeltaTable',
 'List',
 'SparkSession',
 'Window',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 'col',
 'concat_ws',
 'current_timestamp',
 'row_number',
 'spark',
 'transformations']

**import transformations class**

In [0]:
from utils.util_tran import transformations

In [0]:
obj = transformations()

#  **customers Data**

### **read the data**

In [0]:
df_cust = spark.read.table("pysparkdbt.bronze.customers")

In [0]:
df_cust.limit(10).display()

customer_id,first_name,last_name,email,phone_number,city,signup_date,last_updated_timestamp,domain,full_name,process_timestamp
1,Daniel,Reed,azimmerman@ramirez-nelson.com,8117241080,Tiffanyview,2023-12-25,2025-09-15T15:21:05.000Z,ramirez-nelson.com,Daniel Reed,2026-03-22T22:26:34.309Z
10,Megan,Dean,vritter@yahoo.com,0019670618100482,Port Jesse,2023-03-12,2025-09-07T23:05:32.000Z,yahoo.com,Megan Dean,2026-03-22T22:26:34.309Z
100,Gary,Barnes,newmanmelanie@flynn-ross.org,0010575302069,North Charlestown,2022-06-22,2025-09-14T22:47:28.000Z,flynn-ross.org,Gary Barnes,2026-03-22T22:26:34.309Z
101,Stacy,Thomas,hannahwhite@hotmail.com,8882340212190,Lisachester,2020-09-24,2025-09-10T22:45:30.000Z,hotmail.com,Stacy Thomas,2026-03-22T22:26:34.309Z
102,Alan,Rogers,cunninghamjessica@yahoo.com,0010591315078,Harrisbury,2023-07-29,2025-09-19T21:57:29.000Z,yahoo.com,Alan Rogers,2026-03-22T22:26:34.309Z
103,Stephen,Sanders,sherry55@hernandez.com,12586465512757,Port Michael,2022-06-18,2025-09-09T06:06:57.000Z,hernandez.com,Stephen Sanders,2026-03-22T22:26:34.309Z
104,Duane,Bennett,moralessandy@arnold-robinson.com,0018634267566120,Lake Kelly,2020-11-07,2025-09-17T02:22:22.000Z,arnold-robinson.com,Duane Bennett,2026-03-22T22:26:34.309Z
105,Jasmin,Patterson,matthewsbenjamin@lloyd.com,87710366965279,South Wyattfort,2021-04-11,2025-08-25T11:18:42.000Z,lloyd.com,Jasmin Patterson,2026-03-22T22:26:34.309Z
106,Kara,Williams,tyleryoung@gross.org,41127166452578,Seanview,2024-01-05,2025-08-25T04:33:25.000Z,gross.org,Kara Williams,2026-03-22T22:26:34.309Z
107,Martin,Williams,brandyphillips@hotmail.com,17891625036,Melissabury,2021-06-15,2025-09-14T13:56:22.000Z,hotmail.com,Martin Williams,2026-03-22T22:26:34.309Z


### Transformations

**Get the domain name from the email**

In [0]:
df_cust = df_cust.withColumn("domain", split(col('email'), "@")[1])

**normalize the phone number**

In [0]:
df_cust = df_cust.withColumn("phone_number", regexp_replace(col('phone_number'), '[^0-9]', ''))

**add a full name column**

In [0]:
df_cust = df_cust.withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name")))

In [0]:
df_cust.limit(10).display()

customer_id,first_name,last_name,email,phone_number,city,signup_date,last_updated_timestamp,domain,full_name
1,Daniel,Reed,azimmerman@ramirez-nelson.com,8117241080,Tiffanyview,2023-12-25,2025-09-15T15:21:05.000Z,ramirez-nelson.com,Daniel Reed
2,Jonathan,Hansen,williammiller@serrano-jones.com,63219040270198,Baldwinburgh,2021-01-23,2025-09-19T20:19:41.000Z,serrano-jones.com,Jonathan Hansen
3,Samuel,Rodriguez,troy07@carrillo-webb.info,6760940716,New Ashley,2025-05-31,2025-09-13T03:45:53.000Z,carrillo-webb.info,Samuel Rodriguez
4,Victor,Sanchez,jonthompson@thomas.biz,9609292694,Stephanieton,2024-08-31,2025-08-23T00:38:03.000Z,thomas.biz,Victor Sanchez
5,Sabrina,Black,patricknewman@williams.biz,1621699945879470,South Christopherport,2023-05-18,2025-08-20T16:08:38.000Z,williams.biz,Sabrina Black
6,Melissa,Blair,courtney29@gmail.com,332469281291807,East Pamela,2025-09-17,2025-08-24T09:44:20.000Z,gmail.com,Melissa Blair
7,Jacqueline,Williams,robertmoore@gmail.com,7312603248,Fritzmouth,2023-05-29,2025-08-28T19:40:31.000Z,gmail.com,Jacqueline Williams
8,Christina,Arnold,parkerbridget@yahoo.com,8786821357,North Sarah,2023-09-21,2025-09-06T20:36:02.000Z,yahoo.com,Christina Arnold
9,Lisa,Shelton,cody49@johnson.org,9349468560,South Kyleview,2022-12-13,2025-09-08T14:57:58.000Z,johnson.org,Lisa Shelton
10,Megan,Dean,vritter@yahoo.com,0019670618100482,Port Jesse,2023-03-12,2025-09-07T23:05:32.000Z,yahoo.com,Megan Dean


**Deduplication**

In [0]:
df_cust = obj.dedup(df_cust, ['customer_id'], 'last_updated_timestamp')

### **write Data**

**Upsert into Delta table (Silver layer)**

In [0]:
df_cust = obj.process_timestamp(df_cust)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.customers"):
    df_cust.write.format('delta')\
                    .mode('append')\
                    .saveAsTable('pysparkdbt.silver.customers')

else:
    obj.upsert(df_cust, ['customer_id'], 'customers', 'last_updated_timestamp')

In [0]:
%sql
SELECT count(*) FROM pysparkdbt.silver.customers

count(*)
200


#  **drivers Data**

### **read the data**

In [0]:
df_driver = spark.read.table('pysparkdbt.bronze.drivers')

In [0]:
df_driver.limit(10).display()

driver_id,first_name,last_name,phone_number,vehicle_id,driver_rating,city,last_updated_timestamp
1,Latasha,Lopez,262-924-2955x590,1,4.7,East Dorothy,2025-08-25T06:36:26.000Z
2,Alan,Wiley,0967969634,2,3.98,West Susan,2025-09-14T00:44:57.000Z
3,James,Taylor,424-614-1847,3,3.66,Mcintoshton,2025-08-26T22:28:17.000Z
4,Theresa,Benson,617-017-0101x91777,4,3.86,North Courtneychester,2025-09-01T11:40:55.000Z
5,Karen,Jensen,611-060-5683,5,4.87,Brownburgh,2025-09-04T16:35:04.000Z
6,Debra,Smith,556.480.9096x439,6,4.26,Port Williamland,2025-08-31T14:31:37.000Z
7,Justin,Peters,+1-798-568-6952x9778,7,3.67,West Erinborough,2025-09-17T05:57:45.000Z
8,Todd,Young,706.321.8390x08097,8,4.9,Lake Stephen,2025-08-26T06:45:51.000Z
9,Mary,Young,(172)791-0504x5499,9,4.5,West Lindsey,2025-08-27T13:04:32.000Z
10,Jacob,Mack,(509)613-4480,10,4.04,Lauraland,2025-09-09T05:50:23.000Z


### **transformations**

**replace first and last name columns with full name column**

In [0]:
df_driver = df_driver.withColumn('full_name', concat_ws(' ', col('first_name'), col('last_name')))
df_driver = df_driver.drop("first_name","last_name")

**normalize phone number**

In [0]:
df_driver = df_driver.withColumn('phone_number', regexp_replace(col('phone_number'), '[^0-9]', ''))

**Deduplication**

In [0]:
df_driver = obj.dedup(df_driver, ['driver_id'], 'last_updated_timestamp')

In [0]:
df_driver.limit(10).display()

driver_id,phone_number,vehicle_id,driver_rating,city,last_updated_timestamp,full_name
1,2629242955590,1,4.7,East Dorothy,2025-08-25T06:36:26.000Z,Latasha Lopez
10,5096134480,10,4.04,Lauraland,2025-09-09T05:50:23.000Z,Jacob Mack
11,1430778973,11,3.58,Lake Stephenview,2025-08-20T18:43:00.000Z,Matthew Barajas
12,6948696563,12,4.91,New Michaelview,2025-08-25T18:14:33.000Z,Kimberly Sampson
13,3263292468,13,4.52,Strongfort,2025-09-09T09:45:17.000Z,David Silva
14,7192493764,14,4.83,South Tina,2025-09-18T20:52:07.000Z,Sean Christensen
15,2055683393,15,4.38,Petersenfurt,2025-09-01T07:01:10.000Z,Vanessa Williams
16,0012670315560,16,3.53,New Gabrielleside,2025-08-22T03:44:26.000Z,Ryan Henderson
17,6441682978787,17,3.83,Sydneystad,2025-08-30T15:09:25.000Z,Alfred Clayton
18,0159120292438,18,4.12,New Michaelshire,2025-09-13T08:53:24.000Z,Lisa Duarte


### **write Data**

**Upsert into Delta table (Silver layer)**

In [0]:
df_driver = obj.process_timestamp(df_driver)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.drivers"):
    df_driver.write.format('delta')\
                    .mode('append')\
                    .saveAsTable('pysparkdbt.silver.drivers')

else:
    obj.upsert(df_driver, ['driver_id'], 'drivers', 'last_updated_timestamp')

In [0]:
%sql
SELECT count(*) FROM pysparkdbt.silver.drivers

count(*)
50


#  **locations Data**

### **read the data**

In [0]:
df_loc = spark.read.table('pysparkdbt.bronze.locations')

In [0]:
df_loc.limit(10).display()

location_id,city,state,country,latitude,longitude,last_updated_timestamp
1,Lake Davidport,Hawaii,Dominica,34.5682,151.8097,2025-09-03T13:37:24.000Z
2,Mccarthybury,Minnesota,Dominican Republic,71.6928,-162.2836,2025-09-10T10:54:18.000Z
3,Bellhaven,Arkansas,Japan,-53.3075,69.1475,2025-09-17T06:52:51.000Z
4,Moorechester,New Hampshire,Togo,77.4281,57.0052,2025-08-27T19:04:27.000Z
5,Glennview,North Carolina,Gambia,41.1122,-162.4836,2025-08-20T17:31:06.000Z
6,Davilaville,Arizona,Antarctica (the territory South of 60 deg S),64.3413,-3.912,2025-09-05T13:22:58.000Z
7,Shawnfurt,New Jersey,Lithuania,71.1198,-143.6791,2025-09-03T20:35:10.000Z
8,Bradleytown,Florida,Monaco,30.0895,88.091,2025-08-29T19:45:05.000Z
9,East Miguel,Maryland,French Guiana,-52.9274,-56.9343,2025-08-27T23:43:41.000Z
10,Masonside,Wyoming,Saint Lucia,84.9905,81.5852,2025-09-06T20:25:27.000Z


### **transformations**

**Deduplication**

In [0]:
df_loc = obj.dedup(df_loc, ['location_id'], 'last_updated_timestamp')

### **write data**

**Upsert into Delta table (Silver layer)**

In [0]:
df_loc = obj.process_timestamp(df_loc)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.locations"):
    df_loc.write.format('delta')\
                    .mode('append')\
                    .saveAsTable('pysparkdbt.silver.locations')

else:
    obj.upsert(df_loc, ['location_id'], 'locations', 'last_updated_timestamp')

In [0]:
%sql
SELECT count(*) FROM pysparkdbt.silver.locations

count(*)
50


#  **payments Data**

### **read the data**

In [0]:
df_pay = spark.read.table('pysparkdbt.bronze.payments')

In [0]:
df_pay.limit(10).display()

payment_id,trip_id,customer_id,payment_method,payment_status,amount,transaction_time,last_updated_timestamp
1,274,126,Cash,Success,38.15,2025-09-17T13:00:12.000Z,2025-08-30T13:40:53.000Z
2,676,131,Cash,Success,52.07,2025-08-14T13:00:12.000Z,2025-09-08T18:21:05.000Z
3,919,132,Card,Failed,55.5,2025-07-27T13:00:12.000Z,2025-08-21T20:24:08.000Z
4,247,34,Wallet,Pending,28.78,2025-07-27T13:00:12.000Z,2025-08-27T15:03:09.000Z
5,386,62,Card,Failed,55.02,2025-08-01T13:00:12.000Z,2025-09-08T23:15:06.000Z
6,834,78,Wallet,Success,27.43,2025-08-25T13:00:12.000Z,2025-09-20T09:03:05.000Z
7,348,144,Wallet,Failed,74.94,2025-08-26T13:00:12.000Z,2025-08-30T19:11:26.000Z
8,558,82,Card,Failed,19.58,2025-09-09T13:00:12.000Z,2025-09-06T05:53:26.000Z
9,260,151,Card,Pending,24.4,2025-07-23T13:00:12.000Z,2025-09-02T17:43:14.000Z
10,133,70,Cash,Failed,69.99,2025-07-31T13:00:12.000Z,2025-09-11T10:30:16.000Z


### **transformations**

**online payment status column**

In [0]:
df_pay = df_pay.withColumn("online_payment_status",
                             when((col("payment_method") == "Card") & (col("payment_status") == "Success"),"online-success")
                             .when((col("payment_method") == "Card") & (col("payment_status") == "Failed"),"online-failed")
                             .when((col("payment_method") == "Card") & (col("payment_status") == "Pending"),"online-pending")
                             .otherwise("offline")
)

**Deduplication**

In [0]:
df_pay = obj.dedup(df_pay, ['payment_id'], 'last_updated_timestamp')

In [0]:
df_pay.limit(10).display()

payment_id,trip_id,customer_id,payment_method,payment_status,amount,transaction_time,last_updated_timestamp,online_payment_status
1,274,126,Cash,Success,38.15,2025-09-17T13:00:12.000Z,2025-08-30T13:40:53.000Z,offline
10,133,70,Cash,Failed,69.99,2025-07-31T13:00:12.000Z,2025-09-11T10:30:16.000Z,offline
100,265,92,Wallet,Success,90.93,2025-07-15T13:00:12.000Z,2025-09-01T15:06:16.000Z,offline
1000,903,43,Card,Pending,96.61,2025-08-29T13:00:12.000Z,2025-08-30T01:59:42.000Z,online-pending
101,761,107,Cash,Success,8.75,2025-08-10T13:00:12.000Z,2025-09-01T13:16:51.000Z,offline
102,139,134,Wallet,Pending,59.42,2025-07-04T13:00:12.000Z,2025-08-25T19:45:46.000Z,offline
103,604,80,Card,Success,45.54,2025-09-17T13:00:12.000Z,2025-09-13T02:33:29.000Z,online-success
104,434,122,Cash,Failed,73.37,2025-07-04T13:00:12.000Z,2025-09-04T19:07:43.000Z,offline
105,168,80,Cash,Pending,68.93,2025-08-30T13:00:12.000Z,2025-08-30T18:12:56.000Z,offline
106,814,87,Card,Failed,29.59,2025-08-02T13:00:12.000Z,2025-09-19T08:05:12.000Z,online-failed


### **write data**

**Upsert into Delta table (Silver layer)**

In [0]:
df_pay = obj.process_timestamp(df_pay)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.payments"):
    df_pay.write.format('delta')\
                    .mode('append')\
                    .saveAsTable('pysparkdbt.silver.payments')

else:
    obj.upsert(df_pay, ['payment_id'], 'payments', 'last_updated_timestamp')

In [0]:
%sql
SELECT count(*) FROM pysparkdbt.silver.payments

count(*)
1000


# vehicles Data

### **read the data**

In [0]:
df_veh = spark.read.table('pysparkdbt.bronze.vehicles')

In [0]:
df_veh.limit(10).display()

vehicle_id,license_plate,model,make,year,vehicle_type,last_updated_timestamp
1,NXT-8646,Message,"Francis, Smith and Lee",2023,Hatchback,2025-09-02T06:05:20.000Z
2,03S R43,Region,Lawson Group,2017,Sedan,2025-08-31T05:55:09.000Z
3,SKO H06,Prepare,"Moreno, Ruiz and Barker",2023,Luxury,2025-08-30T05:07:29.000Z
4,5R235,Pattern,"Welch, Martinez and Hendricks",2019,Van,2025-09-09T05:52:10.000Z
5,925A,Process,Rivera-Anderson,2014,Hatchback,2025-08-27T11:42:16.000Z
6,6-1130V,On,Brown Ltd,2014,SUV,2025-09-06T06:37:52.000Z
7,L25-TWN,Plan,"Gonzalez, Rios and Rios",2020,Sedan,2025-09-15T13:14:48.000Z
8,GWY 958,Month,"Smith, Mckenzie and Bullock",2017,Sedan,2025-09-11T16:20:12.000Z
9,4FG 919,Speak,"Rice, Barnes and Hernandez",2019,SUV,2025-09-11T02:43:13.000Z
10,6FD 648,Religious,Schwartz and Sons,2017,SUV,2025-09-11T19:10:01.000Z


### **Transformations**

In [0]:
df_veh = df_veh.withColumn('make', upper(col("make")))

**Deduplication**

In [0]:
df_veh = obj.dedup(df_veh, ['vehicle_id'], 'last_updated_timestamp')


### **write Data**

**Upsert into Delta table (Silver layer)**

In [0]:
df_veh = obj.process_timestamp(df_veh)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.vehicles"):
    df_veh.write.format('delta')\
                    .mode('append')\
                    .saveAsTable('pysparkdbt.silver.vehicles')

else:
    obj.upsert(df_veh, ['vehicle_id'], 'vehicles', 'last_updated_timestamp')

In [0]:
%sql
SELECT count(*) FROM pysparkdbt.silver.vehicles

count(*)
50
